In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import *

from network.layer import Layer

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0,IsNew32=False)
chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=100,adc_last_gap=10)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(20, 20,delay3=50)
chip.add_compiler("../compiler/code/")

In [ ]:
# chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
# # deviceType参数：0为ReRAM，1为ECRAM
# # IsNew32参数：False为v1版本，True为v2版本
# chip.set_device_cfg(deviceType=0,IsNew32=True)
# chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=20,adc_last_gap=10)
# chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
# chip.clk_manager.set_cyc(10, 10,delay3=0)
# chip.add_compiler("../compiler/code/")
# # chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
read_times = 201

In [ ]:
crossbar = np.ones((256,256))
for i in range(200):
    v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=0,row_type=0,col_type=0)
    plot_cond(c,vmax=1000,title = f"read_times = {read_times}",path=f"../temp/chip4/times={read_times}")
    read_times = read_times+1

In [ ]:
select = SELECT()

In [ ]:
reset_times = 0

In [ ]:
select.Reset(chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=51,start_v=0.5,delta_v=0.05,tg=5,threshold=200,
    reset_pulse_width=100e-6,sub_base=True,plot_cond=plot_cond,vmax=1400,path=f"../temp/chip4/reset/times={reset_times}")
    

In [ ]:
def Reset(self,chip,need_read,write_times,start_v,delta_v,tg,threshold,reset_pulse_width,sub_base=True,vmax=1000,plot_cond=None,reset_times=0):
    cond_all = np.zeros_like(need_read,dtype=float)
    for i in range(write_times):
        print(f"write_time = {i}")
        v = start_v+i*delta_v
        _,cond,_ = chip.read4(crossbar=need_read,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=sub_base,from_row=True,split_type=0,row_type=0,col_type=0)

        condition_reset = (cond>threshold)&need_read
        cond_all[need_read] = cond[need_read]
        need_read = condition_reset
        
        if plot_cond: plot_cond(cond_all,title=f"v={v:.2f}-needReset={np.sum(condition_reset)}",vmax=vmax,path=f"../temp/chip4/reset/times={reset_times+i}")

        chip.write4(crossbar=condition_reset,row_index=None,col_index=None,write_voltage=v,tg=tg,pulse_width=reset_pulse_width,set_device=False,split_type=0,row_type=0,col_type=0)


In [ ]:
Reset(select,chip=chip,need_read = np.ones((256,256),dtype=bool),write_times=51,start_v=0.5,delta_v=0.05,tg=5,threshold=200,
    reset_pulse_width=100e-6,sub_base=True,plot_cond=plot_cond,vmax=1400,reset_times=1)